In [5]:
%pip install numpy pandas duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.9 MB/s  0:00:006m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import duckdb

conn = duckdb.connect("main.db")
conn.execute("CREATE TABLE IF NOT EXISTS air_quality AS " \
            "SELECT * FROM '/workspaces/Product-Market-Fit-Analysis---Case-Study/Dataset/day-wise-state-wise-air-quality-index-aqi-of-major-cities-and-towns-in-india.csv'")

conn.execute("SELECT * FROM air_quality LIMIT 5").fetchdf()

,date,state,area,number_of_monitoring_stations,prominent_pollutants,aqi_value,air_quality_status,unit,note
0,2025-06-19,Uttar Pradesh,Agra,5.0,"O3,PM2.5,PM10",49.0,Good,number_of_monitoring_stations in Absolute Numb...,None
1,2025-06-19,Karnataka,Bagalkot,1.0,PM10,46.0,Good,number_of_monitoring_stations in Absolute Numb...,None
2,2025-06-19,Maharashtra,Akola,1.0,PM10,26.0,Good,number_of_monitoring_stations in Absolute Numb...,None
3,2025-06-19,Rajasthan,Alwar,1.0,CO,76.0,Satisfactory,number_of_monitoring_stations in Absolute Numb...,None
4,2025-06-19,Andhra Pradesh,Amaravati,1.0,PM10,66.0,Satisfactory,number_of_monitoring_stations in Absolute Numb...,None


In [2]:
conn.execute("""
CREATE TABLE IF NOT EXISTS disease_data AS
SELECT *
FROM read_csv(
    '/workspaces/Product-Market-Fit-Analysis---Case-Study/Dataset/master-data-state-district-and-disease-wise-cases-and-death-reported-due-to-outbreak-of-diseases-as-per-weekly-reports-under-idsp.csv',
    ignore_errors=true
             )
""")
conn.execute("SELECT * FROM disease_data LIMIT 5").fetchdf()

,year,week,outbreak_starting_date,reporting_date,state,district,disease_illness_name,status,cases,deaths,unit,note
0,2025,14,2025-04-05,2025-04-05,Assam,Biswanath,Food Poisoning,Reported,18,0,"cases in absolute number, deaths in absolute n...",None
1,2025,14,2025-04-03,2025-04-04,Bihar,Aurangabad,Fever with Rash,Reported,21,0,"cases in absolute number, deaths in absolute n...",None
2,2025,14,2025-03-31,2025-04-04,Bihar,Madhubani,Chickenpox,Reported,14,0,"cases in absolute number, deaths in absolute n...",None
3,2025,14,2025-04-03,2025-04-04,Gujarat,Bhavnagar,Food Poisoning,Reported,20,0,"cases in absolute number, deaths in absolute n...",None
4,2025,14,2025-04-01,2025-04-01,Gujarat,Vadodara,Acute Diarrheal Disease,Reported,20,0,"cases in absolute number, deaths in absolute n...",None


In [3]:
conn.execute("""
CREATE TABLE IF NOT EXISTS vehicle_registrations AS
SELECT *
FROM read_csv(
    '/workspaces/Product-Market-Fit-Analysis---Case-Study/Dataset/master-data-state-vehicle-class-and-fuel-type-wise-total-number-of-vehicles-registered-in-each-month-in-india.csv',
    ignore_errors=true
             )
""")
conn.execute("SELECT * FROM vehicle_registrations LIMIT 5").fetchdf()

,year,month,state,rto,vehicle_class,fuel,value,unit,note
0,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,BUS,DIESEL,2,value in Absolute Number,None
1,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,GOODS CARRIER,DIESEL,23,value in Absolute Number,None
2,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,GOODS CARRIER,PETROL,1,value in Absolute Number,None
3,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,M-CYCLE/SCOOTER,ELECTRIC(BOV),1,value in Absolute Number,None
4,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,M-CYCLE/SCOOTER,PETROL,387,value in Absolute Number,None


In [4]:
conn.execute("""
CREATE TABLE IF NOT EXISTS population_data AS
SELECT *
FROM read_xlsx(
    '/workspaces/Product-Market-Fit-Analysis---Case-Study/Dataset/population-projection-of-india-state-and-gender-wise-yearly-projected-urban-population-2011-2036.xlsx',
    ignore_errors=true
             )
""")
conn.execute("SELECT * FROM population_data LIMIT 5").fetchdf()

,year,month,state,gender,value,unit,note
0,2036.0,October,West Bengal,Total,43964.0,value in Thousands,NaN
1,2036.0,October,West Bengal,Male,22615.0,value in Thousands,NaN
2,2036.0,October,West Bengal,Female,21349.0,value in Thousands,NaN
3,2036.0,October,Uttarakhand,Total,5506.0,value in Thousands,NaN
4,2036.0,October,Uttarakhand,Male,2922.0,value in Thousands,NaN


In [9]:
"""
Q1. List the top 5 and bottom 5 areas with highest average AQI. 
(Consider areas which contains data from last 6 months: December 2024 to May 2025)
"""

conn.execute(""" 
             WITH avg_aqi AS (
            SELECT
             area,
             AVG(aqi_value) AS avg_aqi 
            FROM air_quality
            WHERE date >= '2024-12-01' AND date <= '2025-05-31'
            GROUP BY area
             ),

            
             aqi_rankings AS (
             SELECT
             *,
             DENSE_RANK() OVER(ORDER BY avg_aqi DESC) AS highest_aqi_rank,
             DENSE_RANK() OVER(ORDER BY avg_aqi ASC) AS lowest_aqi_rank
             FROM avg_aqi
             )

             SELECT
             area,
             avg_aqi,
             'bottom_5_areas' AS list
             FROM aqi_rankings
             WHERE lowest_aqi_rank <=5 

             UNION ALL

             SELECT
             area,
             avg_aqi,
             'top_5_areas' AS list
             FROM aqi_rankings
             WHERE highest_aqi_rank <=5 

    """).fetchdf()

,area,avg_aqi,list
0,Tirunelveli,33.167742,bottom_5_areas
1,Madikeri,40.239766,bottom_5_areas
2,Palkalaiperur,40.691176,bottom_5_areas
3,Thanjavur,44.326087,bottom_5_areas
4,Chamarajanagar,44.929032,bottom_5_areas
5,Gurugram,197.022599,top_5_areas
6,Hajipur,217.114458,top_5_areas
7,Bahadurgarh,226.437500,top_5_areas
8,Delhi,227.038674,top_5_areas
9,Byrnihat,265.309353,top_5_areas


In [21]:
"""
Q2. List out top 2 and bottom 2 prominent pollutants for each state of southern India. 
(Consider data post covid: 2022 onwards) 
"""
conn.execute(""" 
             WITH filtered_data AS (
            SELECT
             state,
             prominent_pollutants,
             UNNEST(string_split(prominent_pollutants, ',')) AS pollutant
            FROM air_quality
            WHERE date >= '2022-01-01' AND 
             state IN ('Andhra Pradesh', 'Telangana', 'Karnataka', 'Kerala', 'Tamil Nadu')
            ),

            pollutant_counts AS (
                SELECT
                    state,
                    pollutant,
                    COUNT(*) AS cnt
                FROM filtered_data
                GROUP BY state, pollutant
            ),

            ranked_pollutants AS (
                SELECT
                    state,
                    pollutant,
                    cnt,
                    RANK() OVER (PARTITION BY state ORDER BY cnt DESC) AS rank_desc,
                    RANK() OVER (PARTITION BY state ORDER BY cnt ASC)  AS rank_asc
                FROM pollutant_counts
            )

            SELECT
                state,
                pollutant,
                cnt,
                CASE
                    WHEN rank_desc <= 2 THEN 'Top 2'
                    WHEN rank_asc  <= 2 THEN 'Bottom 2'
                END AS category
            FROM ranked_pollutants
            WHERE rank_desc <= 2
            OR rank_asc  <= 2
            ORDER BY state, category, cnt DESC;

    """).fetchdf()

,state,pollutant,cnt,category
0,Andhra Pradesh,NO2,263,Bottom 2
1,Andhra Pradesh,SO2,11,Bottom 2
2,Andhra Pradesh,PM10,3964,Top 2
3,Andhra Pradesh,PM2.5,2599,Top 2
4,Karnataka,NH3,34,Bottom 2
5,Karnataka,SO3,1,Bottom 2
6,Karnataka,PM10,16231,Top 2
7,Karnataka,CO,3646,Top 2
8,Kerala,NH3,10,Bottom 2
9,Kerala,SO2,8,Bottom 2


In [29]:
"""
Q3. Does AQI improve on weekends vs weekdays in Indian metro cities 
(Delhi, Mumbai, Chennai, Kolkata, Bengaluru, Hyderabad, Ahmedabad, Pune)? 
(Consider data from last 1 year)

-- Day of the week (Sunday = 0, Saturday = 6)
"""
conn.execute(""" 
            
            WITH filtered_data AS (
            SELECT
             CASE
                WHEN EXTRACT('dayofweek' FROM date) = 0 OR EXTRACT('dayofweek' FROM date) = 6
                THEN 'Weekend'
                ELSE 'Weekday'
             END AS part_of_week,
             area,
             aqi_value
            FROM air_quality
            WHERE date >= '2024-01-01' AND date <= '2024-12-31'
             AND 
             area IN ('Delhi', 'Mumbai', 'Chennai', 'Kolkata', 'Bengaluru', 'Hyderabad', 'Ahmedabad', 'Pune')
            ),
            
            aggregated_data AS (
             SELECT
                area,
                part_of_week,
                AVG(aqi_value) AS avg_aqi
             FROM filtered_data
                GROUP BY area, part_of_week
             ORDER BY area
             )
             
             SELECT 
                weekday.area,
                weekday.avg_aqi AS weekday_avg_aqi,
                weekend.avg_aqi AS weekend_avg_aqi,
                CASE 
                    WHEN weekday.avg_aqi > weekend.avg_aqi THEN 'AQI Improved on Weekend'
                    WHEN weekday.avg_aqi < weekend.avg_aqi THEN 'AQI Worsened on Weekend'
                END AS result
             FROM aggregated_data weekday
             JOIN aggregated_data weekend
             ON weekday.area = weekend.area
             WHERE weekday.part_of_week = 'Weekday' AND weekend.part_of_week = 'Weekend' 

    """).fetchdf()


,area,weekday_avg_aqi,weekend_avg_aqi,result
0,Ahmedabad,114.893130,113.317308,AQI Improved on Weekend
1,Bengaluru,73.965649,73.865385,AQI Improved on Weekend
2,Chennai,72.194656,70.980769,AQI Improved on Weekend
3,Delhi,209.477099,207.567308,AQI Improved on Weekend
4,Hyderabad,76.744275,76.894231,AQI Worsened on Weekend
5,Kolkata,101.007634,100.480769,AQI Improved on Weekend
6,Mumbai,90.648855,92.875000,AQI Worsened on Weekend
7,Pune,96.702290,96.269231,AQI Improved on Weekend


In [40]:
"""
Q4. Which months consistently show the worst air quality across Indian states — 
(Consider top 10 states with high distinct areas) 
"""

conn.execute(""" 
WITH top_states AS (
    SELECT
        state,
        COUNT(DISTINCT area) AS num_of_areas
    FROM air_quality
    GROUP BY state
    ORDER BY num_of_areas DESC
    LIMIT 10
),

monthly_state_aqi AS (
    SELECT
        EXTRACT(MONTH FROM date) AS month,
        aq.state,
        AVG(aqi_value) AS avg_aqi
    FROM air_quality aq
    JOIN top_states ts
        ON aq.state = ts.state
    GROUP BY month, aq.state
),

monthly_overall_aqi AS (
    SELECT
        month,
        AVG(avg_aqi) AS overall_avg_aqi
    FROM monthly_state_aqi
    GROUP BY month
)

SELECT
    month,
    overall_avg_aqi
FROM monthly_overall_aqi
ORDER BY overall_avg_aqi DESC;             
    """).fetchdf()


,month,overall_avg_aqi
0,11,166.611904
1,12,164.017378
2,1,161.899580
3,2,134.683294
4,3,119.140584
5,4,116.155133
6,10,113.945600
7,5,104.045361
8,6,91.471072
9,9,68.053210


In [43]:
"""
Q5. For the city of Bengaluru, how many days fell under each air quality category 
(e.g., Good, Moderate, Poor, etc.) between March and May 2025?  
"""

conn.execute(""" 

SELECT
    area,
    air_quality_status,
    COUNT(*) AS cnt  
FROM air_quality
WHERE area = 'Bengaluru' AND date >= '2025-03-01' AND date <= '2025-05-01'
GROUP BY area, air_quality_status
ORDER BY cnt DESC
             
""").fetchdf()


,area,air_quality_status,cnt
0,Bengaluru,Satisfactory,49
1,Bengaluru,Moderate,13


In [54]:
"""
Q6. List the top two most reported disease illnesses in each state over the past three 
years, along with the corresponding average Air Quality Index (AQI) for that period.
"""

conn.execute(""" 
WITH diseases_data_aggregated AS (        
        SELECT 
            state,
            disease_illness_name,
            COUNT(*) AS cases
        FROM disease_data
        WHERE year >= 2022 AND year <= 2025
        GROUP BY state, disease_illness_name
        ),
        
        diseases_data_ranked AS (
        SELECT
             *,
             DENSE_RANK() OVER(PARTITION BY state ORDER BY cases DESC) AS rnk
        FROM diseases_data_aggregated     
        ),
        
        top2_diseases_per_state AS (
        SELECT
            state,
            disease_illness_name,
            cases
        FROM diseases_data_ranked
            WHERE rnk <= 2
        ),
        
        avg_aqi_per_state AS (
        SELECT
            state,
            AVG(aqi_value) AS avg_aqi
        FROM air_quality
        WHERE date BETWEEN '2022-01-01' AND '2024-12-31'
        GROUP BY state    
        )
             
        SELECT 
            avg_aqi_per_state.state,
            top2_diseases_per_state.disease_illness_name,
            top2_diseases_per_state.cases,
            avg_aqi_per_state.avg_aqi
        FROM avg_aqi_per_state
        LEFT JOIN top2_diseases_per_state
        ON avg_aqi_per_state.state = top2_diseases_per_state.state
             
""").fetch_df()

,state,disease_illness_name,cases,avg_aqi
0,Bihar,Fever with Rash,62,165.439666
1,Bihar,Acute Diarrheal Disease,52,165.439666
2,Rajasthan,Acute Diarrheal Disease,19,128.364710
3,Rajasthan,Food Poisoning,10,128.364710
4,Assam,Acute Diarrheal Disease,88,112.077610
...,...,...,...,...
68,Manipur,Acute Diarrheal Disease,4,100.558077
69,Jammu and Kashmir,Hepatitis A,68,77.120459
70,Jammu and Kashmir,Mumps,32,77.120459
71,Kerala,Food Poisoning,196,68.732424


In [ ]:
"""
Q7. List the top 5 states with high EV adoption and analyse if their average AQI is 
significantly better compared to states with lower EV adoption
"""

conn.execute("""
        SELECT
            state,
            
        FROM vehicle_registrations
        WHERE fuel LIKE '% EV%' AND year >= 2022 AND year < 2025
             


""").fetch_df()

,state
0,Andaman and Nicobar Islands
1,Andaman and Nicobar Islands
2,Andaman and Nicobar Islands
3,Andaman and Nicobar Islands
4,Andhra Pradesh
...,...
2020,West Bengal
2021,West Bengal
2022,West Bengal
2023,West Bengal
